### Importation des bibliothèques

Les bibliothèques nécessaires sont importées afin de manipuler les données et réaliser les visualisations.

In [ ]:
%pip install numpy pandas pandas seaborn 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### Chargement des données

Le fichier `mesures_capteurs.csv` est importé dans un DataFrame nommé `df`. Ce DataFrame sera utilisé tout au long de l'atelier.

In [ ]:
df = pd.read_csv("../data/mesures_capteurs.csv")

df

### Exploration des données

Une première inspection du DataFrame permet de vérifier que les données ont été correctement chargées et d'observer leur structure.

In [ ]:
df.head()

df.info()

df.describe()

df.isnull().sum()

### 1.1 : Vérification des doublons

Avant de poursuivre l'analyse des données, il est important de vérifier si le DataFrame contient des lignes dupliquées. Les doublons peuvent fausser les statistiques et les visualisations réalisées par la suite.

In [ ]:
# Nombre de lignes dupliquées
nb_doublons = df.duplicated().sum()

print(f"Nombre de doublons : {nb_doublons}")

### 1.2 : Suppression des doublons

Si des doublons sont présents, ils sont supprimés afin de conserver une seule occurrence de chaque observation. Une nouvelle vérification est ensuite réalisée pour confirmer que le nettoyage a bien été effectué.

In [ ]:
# Suppression des doublons
df = df.drop_duplicates()

# Vérification après suppression
nb_doublons = df.duplicated().sum()

print(f"Nombre de doublons après suppression : {nb_doublons}")

### 2.1 : Définition de la cible et des caractéristiques

Dans cette étape, la colonne `etat` est définie comme la variable cible (`y`), c'est-à-dire la valeur que le modèle devra prédire.

Les colonnes `temperature`, `humidite`, `pression` et `consommation` constituent les variables explicatives (`X`) qui serviront à entraîner le modèle.

In [ ]:
# Variable cible
y = df["etat"]

# Variables explicatives
X = df[["temperature", "humidite", "pression", "consommation"]]

### 2.2 : Aperçu des données

Les cinq premières lignes de `X` et de `y` sont affichées afin de vérifier que les variables ont été correctement sélectionnées.

In [ ]:
print("Variables explicatives (X) :")
display(X.head())

print("Variable cible (y) :")
display(y.head())

### 2.3 : Type du problème

La variable cible `etat` contient des catégories (par exemple : `OK`, `ALERTE` et `ERREUR`). Le modèle devra prédire une classe à partir de plusieurs caractéristiques.

Il s'agit donc d'un problème de **classification**.

In [ ]:
print("Type du problème :", "Classification")

print("\nClasses présentes dans la cible :")
print(y.unique())

### 3.1 : Découpage Train/Test

Les données sont divisées en deux ensembles :

- un ensemble d'entraînement (`train`) utilisé pour apprendre le modèle ;
- un ensemble de test (`test`) utilisé pour évaluer ses performances.

Le découpage est réalisé en réservant 20 % des données au test. La reproductibilité est assurée grâce à `random_state`, tandis que `stratify` permet de conserver la même répartition des classes dans les deux ensembles.

In [ ]:
# Vérifier les stats de NaN

df.isnull().sum()

In [ ]:
# suppresion des valeurs manquantes

df = df.dropna()

# Redéfinir X et y
X = df[["temperature", "humidite", "pression", "consommation"]]
y = df["etat"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

### 3.2 : Vérifier du découpage

Les dimensions des ensembles obtenus sont affichées afin de vérifier que le découpage a été correctement réalisé.

In [ ]:
print("Dimensions de X_train :", X_train.shape)
print("Dimensions de X_test  :", X_test.shape)

print("Dimensions de y_train :", y_train.shape)
print("Dimensions de y_test  :", y_test.shape)

### 4.1 : Vérifier des valeurs manquantes

Avant d'entraîner un modèle de Machine Learning, il est important de vérifier si les variables explicatives contiennent des valeurs manquantes. Ces valeurs devront être traitées afin d'éviter des erreurs lors de l'entraînement du modèle.

In [ ]:
# Nombre de valeurs manquantes par variable
X_train.isnull().sum()

In [ ]:
print("Valeurs manquantes dans X_train :")
print(X_train.isnull().sum())

print("\nValeurs manquantes dans X_test :")
print(X_test.isnull().sum())

### 4.2 : Sélection de l'imputeur

L'imputeur `SimpleImputer` est utilisé pour remplacer automatiquement les valeurs manquantes. La stratégie choisie est la médiane.

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

### 4.3 : Choix de la médiane

La médiane est moins sensible aux valeurs extrêmes que la moyenne. Elle permet donc de remplacer les valeurs manquantes sans être fortement influencée par d'éventuelles observations atypiques présentes dans les données.

### 4.4 : Apprentissage de l'imputeur

L'imputeur est ajusté uniquement sur les données d'entraînement (`X_train`). Il calcule la médiane de chaque variable afin de remplacer les valeurs manquantes.

In [48]:
imputer.fit(X_train)

# Médianes calculées
print(imputer.statistics_)

[  24.905   64.98  1012.47   206.99 ]


### 4.5 : Remplacement des valeurs manquantes

Les ensembles d'entraînement et de test sont transformés à l'aide de l'imputeur. Les valeurs manquantes sont remplacées par les médianes calculées sur `X_train`.

In [ ]:
X_train_imputed = imputer.transform(X_train)
X_test_imputed = imputer.transform(X_test)

print(X_train_imputed, X_test_imputed)

[[  22.85   79.61 1004.14  139.74]
 [  20.88   40.87 1025.89  294.72]
 [  23.94   56.67 1000.28   56.72]
 ...
 [  21.45   71.66 1021.68  196.49]
 [  25.91   49.2  1016.57  201.37]
 [  26.14   84.97 1003.59  142.31]] [[  23.27   67.28 1033.14  234.46]
 [  20.39   66.77 1022.7   135.43]
 [  33.29   50.43 1009.87  356.76]
 [  24.12   71.32 1001.77  164.9 ]
 [  25.36   69.99  996.14  228.06]
 [  23.16   65.   1007.23  164.24]
 [  24.52   73.93 1006.8   202.15]
 [  21.85   71.61 1007.87  160.41]
 [  23.89   64.88 1011.31  225.28]
 [  22.35   62.91 1016.12  213.05]
 [  31.51   72.49 1018.21  307.97]
 [  21.46   56.86 1017.93  200.8 ]
 [  22.44   55.56 1023.41  208.5 ]
 [  26.82   57.71 1009.68  308.53]
 [  28.66   35.83 1022.52  142.86]
 [  26.62   60.22 1023.47  207.79]
 [  20.39   85.01 1013.45  161.04]
 [  24.     79.73  993.39  116.2 ]
 [  22.35   61.16 1015.31  138.64]
 [  29.73   65.31 1015.23  280.63]
 [  20.86   78.12 1010.48   62.95]
 [  27.85   58.01 1012.29  208.09]
 [  24.17   47